# RUNTIME

It's critical to change your runtime to a GPU runtime for this notebook to run correctly.

In your toolbar - Runtime - Change Runtime Type - choose either either of the following.

T4 GPU

v5e-1 TPU


> ### Note on Labs and Assignments
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis you must write.
>
> These sections are graded and are not optional.


# Module 1 Lab 1: AI Project Classification and Decision Boundaries

**Notebook:** Student Template  
**Required model:** `gemma3:1b` through Ollama  
**Data:** Fictional cases only

This lab introduces the course's recurring assignment pattern: run a bounded AI task, preserve evidence, independently evaluate the output, and decide what responsibility must remain with a person.


## Learning Objectives

By completing this notebook, you will:

1. Classify business projects by their primary AI approach.
2. Evaluate model reasoning rather than treating it as an answer key.
3. Compare prompts that request unsupported precision or consequential authority.
4. Redesign AI use from decision making to evidence gathering.
5. Connect an AI proposal to a measurable outcome and simpler alternative.


## Important Instructions

1. Read the assignment and Module 1 reading first.
2. Use the fixed `gemma3:1b` model; do not substitute another model.
3. Change code where you see `🔧` and write analysis where you see `🖊`.
4. Run cells from top to bottom and preserve every output.
5. Use only the supplied fictional cases and résumé.
6. Restart the kernel and run all cells before submitting.

The notebook intentionally stops before a model run when required `TODO` code remains.


## Setup Ollama

Run the next two cells. The notebook connects to Ollama and downloads the required model if it is missing. The one-time `gemma3:1b` download is approximately 815 MB.


In [13]:
import subprocess
import time

In [14]:
# Download and install Ollama (Google Colab only — skip if running locally)
install_zstd = subprocess.run(
    "sudo apt-get install zstd",
    shell=True,
    capture_output=True,
    text=True,
)

install_zstd

CompletedProcess(args='sudo apt-get install zstd', returncode=0, stdout='Reading package lists...\nBuilding dependency tree...\nReading state information...\nzstd is already the newest version (1.4.8+dfsg-3build1).\n0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.\n', stderr='')

In [15]:
# RUN THIS CELL. YOU SHOULD SEE Ollama installed and Ollama server is running messages.

# Download and install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError(f"Ollama installation failed:\n{install.stderr}")
print("Ollama installed.")

# Start the Ollama server as a background process
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give the server a few seconds to initialize before any requests are made
time.sleep(3)
print("Ollama server is running.")

Ollama installed.
Ollama server is running.


In [16]:
# RUN THIS CELL
from datetime import datetime
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

from IPython.display import Markdown, display

MODEL_NAME = "gemma3:1b"
OLLAMA_BASE_URL = "http://localhost:11434"
AUTO_PULL_MODEL = True
GENERATION_OPTIONS = {"temperature": 0, "seed": 4490, "num_ctx": 4096}
RUN_TIMESTAMP = datetime.now().astimezone().isoformat(timespec="seconds")


print(f"Required model: {MODEL_NAME}")
print(f"Run date: {RUN_TIMESTAMP}")


Required model: gemma3:1b
Run date: 2026-08-27T19:38:24+00:00


In [17]:
# RUN THIS CELL
def require_finished(label, value):
    if value is None or "TODO" in str(value) or str(value).strip() == "Your Name":
        raise ValueError(f"Complete {label} before running this cell.")


def ollama_request(path, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {details}") from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def chat_once(prompt, timeout=600):
    response = ollama_request(
        "/api/chat",
        {
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": GENERATION_OPTIONS,
        },
        timeout=timeout,
    )
    return response["message"]["content"].strip()


version_info = ollama_request("/api/version")
available_models = {
    item.get("name") or item.get("model")
    for item in ollama_request("/api/tags").get("models", [])
}
print(f"Connected to Ollama {version_info.get('version', 'unknown version')}.")

if MODEL_NAME not in available_models:
    if not AUTO_PULL_MODEL:
        raise RuntimeError(f"{MODEL_NAME} is not installed.")
    print(f"Downloading {MODEL_NAME}. This is a one-time download...")
    ollama_request("/api/pull", {"model": MODEL_NAME, "stream": False}, timeout=3600)

print(f"{MODEL_NAME} is ready.")


Connected to Ollama 0.33.1.
gemma3:1b is ready.


# Part 1: Classify Four AI Projects

Write one prompt that covers all four fictional retailer cases. The provided code uses a response schema only to keep the model output complete and tabular; it does not decide the classifications.


### TODO - INSTRUCT 🔧


In [18]:
# 🔧 TODO - INSTRUCT
# Replace the TODO text with your own prompt. Your prompt must cover all four
# cases and request one primary category, any secondary categories, reasoning,
# and reasonable alternatives.
student_prompt = """

Do the stuff

A. Customer retention	Estimate which subscription customers are likely to cancel in the next 30 days.
B. Online merchandising	Rank products for each website visitor based on behavior and preferences.
C. Warehouse quality	Detect damaged packages from images captured on a conveyor line.
D. Employee support	Answer employee questions and draft responses using approved HR policies.

"""



note, the next cell will likely take 1-2 minutes or so to run. if it takes much longer you likely are still running on a CPU runtime instead of a GPU and should change the runtime as requested in the beginning of the notebook.

In [19]:
# RUN THIS CELL
require_finished("classification prompt", student_prompt)

category_values = [
    "Rules and expert systems",
    "Predictive analytics and forecasting",
    "Classification and anomaly detection",
    "Recommendation and ranking",
    "Optimization",
    "Computer vision",
    "Speech AI",
    "Natural language processing",
    "Generative AI",
    "Agentic AI",
]
case_schema = {
    "type": "object",
    "properties": {
        "primary_category": {"type": "string", "enum": category_values},
        "secondary_categories": {
            "type": "array",
            "items": {"type": "string", "enum": category_values},
        },
        "reasoning": {"type": "string"},
        "reasonable_alternative": {"type": "string"},
    },
    "required": [
        "primary_category",
        "secondary_categories",
        "reasoning",
        "reasonable_alternative",
    ],
    "additionalProperties": False,
}
analysis_schema = {
    "type": "object",
    "properties": {key: case_schema for key in ["A", "B", "C", "D"]},
    "required": ["A", "B", "C", "D"],
    "additionalProperties": False,
}
response = ollama_request(
    "/api/chat",
    {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": student_prompt}],
        "stream": False,
        "format": analysis_schema,
        "options": GENERATION_OPTIONS,
    },
    timeout=600,
)
raw_classification_response = response["message"]["content"].strip()

analysis_data = json.loads(raw_classification_response)
display(Markdown("### Formatted Model Output\n```json\n" + json.dumps(analysis_data, indent=2) + "\n```"))


### Formatted Model Output
```json
{
  "A": {
    "primary_category": "Agentic AI",
    "secondary_categories": [
      "Recommendation and ranking",
      "Predictive analytics and forecasting"
    ],
    "reasoning": "Customer retention is a key metric for businesses. Predicting which customers are likely to cancel is crucial for proactive retention efforts.  A high cancellation rate would indicate a need for intervention.",
    "reasonable_alternative": "The best answer is A.  Customer retention is a primary concern for most businesses."
  },
  "B": {
    "primary_category": "Optimization",
    "secondary_categories": [
      "Agentic AI",
      "Recommendation and ranking"
    ],
    "reasoning": "Online merchandising aims to improve the user experience and drive sales.  Ranking products based on behavior and preferences is a core strategy for optimizing the website and increasing conversions.  This directly impacts revenue and customer satisfaction.",
    "reasonable_alternative": "B is a strong contender, but A is slightly more focused on a critical business outcome."
  },
  "C": {
    "primary_category": "Predictive analytics and forecasting",
    "secondary_categories": [
      "Agentic AI",
      "Optimization"
    ],
    "reasoning": "Detecting damaged packages is a vital quality control process.  This data can be used to identify potential issues and prevent losses.  It's a direct measure of product quality and customer satisfaction.",
    "reasonable_alternative": "C is a logical answer, but C is more about a specific process than a broader business goal."
  },
  "D": {
    "primary_category": "Agentic AI",
    "secondary_categories": [
      "Recommendation and ranking",
      "Optimization"
    ],
    "reasoning": "Employee support is essential for a positive work environment. Answering questions and drafting responses ensures employees can efficiently handle their tasks and maintain a productive workflow.  This is a fundamental operational need.",
    "reasonable_alternative": "D is correct, but it's a foundational operational need."
  }
}
```

In [20]:
# RUN THIS CELL
project_names = {
    "A": "A. Customer retention",
    "B": "B. Online merchandising",
    "C": "C. Warehouse quality",
    "D": "D. Employee support",
}

def markdown_cell(value):
    return str(value).replace("|", "\\|").replace("\n", " ").strip()

table_lines = [
    "| Project | AI's primary classification | Secondary classification, if any | Summary of AI's reasoning |",
    "|---|---|---|---|",
]
for key in "ABCD":
    result = analysis_data[key]
    secondary = ", ".join(result["secondary_categories"]) or "None identified"
    table_lines.append(
        "| "
        + " | ".join(
            markdown_cell(value)
            for value in [
                project_names[key],
                result["primary_category"],
                secondary,
                result["reasoning"],
            ]
        )
        + " |"
    )

classification_table_md = "\n".join(table_lines)
display(Markdown(classification_table_md))


| Project | AI's primary classification | Secondary classification, if any | Summary of AI's reasoning |
|---|---|---|---|
| A. Customer retention | Agentic AI | Recommendation and ranking, Predictive analytics and forecasting | Customer retention is a key metric for businesses. Predicting which customers are likely to cancel is crucial for proactive retention efforts.  A high cancellation rate would indicate a need for intervention. |
| B. Online merchandising | Optimization | Agentic AI, Recommendation and ranking | Online merchandising aims to improve the user experience and drive sales.  Ranking products based on behavior and preferences is a core strategy for optimizing the website and increasing conversions.  This directly impacts revenue and customer satisfaction. |
| C. Warehouse quality | Predictive analytics and forecasting | Agentic AI, Optimization | Detecting damaged packages is a vital quality control process.  This data can be used to identify potential issues and prevent losses.  It's a direct measure of product quality and customer satisfaction. |
| D. Employee support | Agentic AI | Recommendation and ranking, Optimization | Employee support is essential for a positive work environment. Answering questions and drafting responses ensures employees can efficiently handle their tasks and maintain a productive workflow.  This is a fundamental operational need. |

### TODO - REFLECT 🖊

Write **2–4 sentences for each case**. Evaluate at least one specific model claim, use the input/task/output distinction, and challenge unsupported secondary categories.

#### A. Customer retention

🖊 TODO: State agree/partially agree/disagree and support your judgment.

#### B. Online merchandising

🖊 TODO: State agree/partially agree/disagree and support your judgment.

#### C. Warehouse quality

🖊 TODO: State agree/partially agree/disagree and support your judgment.

#### D. Employee support

🖊 TODO: State agree/partially agree/disagree and support your judgment.


# Part 2: Résumé Review and Decision Boundaries

The following job description and résumé are fictional. Run the two fixed prompts without changing them. The purpose is to observe how prompt framing can push a model toward unsupported precision or consequential authority—not to evaluate a real person.


In [21]:
# RUN THIS CELL
JOB_DESCRIPTION = """Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience
"""

FICTIONAL_RESUME = """Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations
"""

display(Markdown("## Fictional Job Description\n```text\n" + JOB_DESCRIPTION + "\n```"))
display(Markdown("## Fictional Résumé\n```text\n" + FICTIONAL_RESUME + "\n```"))


## Fictional Job Description
```text
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience

```

## Fictional Résumé
```text
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

## Compare Two Fixed Decision-Oriented Prompts

Both prompts use the same source material and model. Only the requested output changes.


In [22]:
# RUN THIS CELL
shared_material = (
    "JOB DESCRIPTION\n---------------\n"
    + JOB_DESCRIPTION
    + "\n\nFICTIONAL RESUME\n------------------\n"
    + FICTIONAL_RESUME
)

score_prompt = (
    "Review this applicant for the job. Assign an overall fit score from 0 to 100 "
    "and explain the score.\n\n" + shared_material
)
decision_prompt = (
    "Review this applicant for the job. Make the final decision: HIRE or DO NOT HIRE. "
    "State the decision first, then explain it.\n\n" + shared_material
)

score_output = chat_once(score_prompt)
decision_output = chat_once(decision_prompt)

display(Markdown("### Prompt 1: 0–100 Score\n```text\n" + score_prompt + "\n```"))
display(Markdown("### Model Output 1\n" + score_output))
display(Markdown("### Prompt 2: Hire/Do Not Hire\n```text\n" + decision_prompt + "\n```"))
display(Markdown("### Model Output 2\n" + decision_output))


### Prompt 1: 0–100 Score
```text
Review this applicant for the job. Assign an overall fit score from 0 to 100 and explain the score.

JOB DESCRIPTION
---------------
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience


FICTIONAL RESUME
------------------
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

### Model Output 1
Okay, let's review Jordan Lee’s resume and assign a fit score and explanation.

**Overall Fit Score: 78/100**

**Explanation:**

Jordan’s resume demonstrates a solid foundation for the Operations Analyst role, leaning heavily towards the “Documenting/Improving Business Processes” and “Process-Mapping” aspects. However, it’s slightly lacking in the SQL and advanced spreadsheet skills that are explicitly preferred.  Here’s a breakdown of why it scores high and where it falls short:

**Strengths:**

* **Experience:** Two years of experience documenting and improving processes is a significant positive.  His work on receiving and inventory workflows, dashboards, and procedures demonstrates a practical understanding of process improvement.
* **Excel Proficiency:**  He excels in Excel – pivot tables, formulas, conditional formatting – which is crucial for data analysis and reporting. This is a key requirement.
* **Stakeholder Communication:**  Presenting delay and rework findings to supervisors shows he can effectively communicate information to those who need it.
* **Procedure Writing:**  Creating 14 standard operating procedures is a valuable accomplishment, demonstrating a commitment to standardization and documentation.
* **Process Mapping:**  The process-mapping workshop is a strong indicator of his ability to identify and streamline workflows – a critical skill for an Operations Analyst.

**Weaknesses (Areas for Improvement - impacting fit):**

* **SQL/Process-Mapping Experience:**  The resume doesn't explicitly state he has SQL experience or process-mapping experience. This is a significant gap.  The preferred qualifications suggest a need for these skills.
* **Lack of SQL/Process-Mapping:**  The absence of these skills makes him less competitive for the job description, particularly given the preferred qualifications.  It’s a noticeable omission.

**Recommendations for Improvement (to boost fit):**

* **Highlight SQL/Process-Mapping:**  If possible, subtly weave in examples of his work that demonstrates these skills.  Even a brief mention of a project where he used SQL to analyze data or created a process map would be beneficial.
* **Expand on Process Mapping:**  If he has any experience with process mapping, even in a small way, it would be valuable to mention it.

**Overall, Jordan has a good base.  With a focused effort to add SQL/process-mapping experience, he’ll significantly increase his chances of success in the role.**

---

To give a more precise score, I'd need to see the *actual* content of his SQL/process-mapping experience, but this assessment provides a solid overall picture.

### Prompt 2: Hire/Do Not Hire
```text
Review this applicant for the job. Make the final decision: HIRE or DO NOT HIRE. State the decision first, then explain it.

JOB DESCRIPTION
---------------
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience


FICTIONAL RESUME
------------------
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

### Model Output 2
**HIRE**

**Decision:** Hire

**Explanation:** Jordan Lee possesses the necessary qualifications and experience outlined in the job description. His experience documenting and improving business processes (specifically through his work on receiving and inventory workflows), creating dashboards, presenting findings, and maintaining procedures demonstrates a strong foundation for the Operations Analyst role.  His demonstrated skills in Excel, process mapping, and procedure writing, particularly his experience with pivot tables, conditional formatting, and dashboard creation, are highly relevant.  The preferred qualifications of SQL and process mapping are valuable assets, but Jordan's existing skillset and experience are strong enough to warrant a hire.  The resume clearly highlights his accomplishments and demonstrates a clear fit for the role.

### TODO - REFLECT 🖊

Compare the two outputs.

🖊 TODO: Explain how the requested output format changed the response.

🖊 TODO: Identify evidence the model used, missed, assumed, or invented.

🖊 TODO: Explain why a 0–100 score can look more objective than the evidence supports.

🖊 TODO: Explain why neither output should transfer the employment decision to this model.

🖊 TODO: State what an accountable human reviewer must still do.


## Redesign the Task for Evidence Gathering

Use the same fictional materials, but constrain the model to organize evidence for an accountable human reviewer.


### TODO - INSTRUCT 🔧


In [26]:
# 🔧 TODO - INSTRUCT
# Redesign the task so the model gathers job-relevant evidence without scoring,
# ranking, recommending, shortlisting, or making an employment decision.
evidence_prompt = """

Do the stuff

A. Customer retention	Estimate which subscription customers are likely to cancel in the next 30 days.
B. Online merchandising	Rank products for each website visitor based on behavior and preferences.
C. Warehouse quality	Detect damaged packages from images captured on a conveyor line.
D. Employee support	Answer employee questions and draft responses using approved HR policies.


"""


In [27]:
# RUN THIS CELL
require_finished("evidence-gathering prompt", evidence_prompt)
evidence_output = chat_once(evidence_prompt + "\n\n" + shared_material)
display(Markdown("### Evidence-Gathering Prompt\n```text\n" + evidence_prompt + "\n```"))
display(Markdown("### Evidence-Gathering Output\n" + evidence_output))


### Evidence-Gathering Prompt
```text


Do the stuff

A. Customer retention	Estimate which subscription customers are likely to cancel in the next 30 days.
B. Online merchandising	Rank products for each website visitor based on behavior and preferences.
C. Warehouse quality	Detect damaged packages from images captured on a conveyor line.
D. Employee support	Answer employee questions and draft responses using approved HR policies.



```

### Evidence-Gathering Output
Based on the job description and Jordan Lee’s resume, here’s an analysis of which options are most likely to be relevant and which are less likely:

*   **A. Customer retention:** This is **highly likely**. The resume highlights his experience documenting workflows and processes, which directly relates to improving customer experience and reducing churn.

*   **B. Online merchandising:** This is **less likely**. The job description focuses on operational processes, not directly on website merchandising.

*   **C. Warehouse quality:** This is **possible**, but less likely than A. The resume mentions a process-mapping workshop, which could involve quality control and identifying potential issues.

*   **D. Employee support:** This is **unlikely**. The resume emphasizes documentation and procedures, not direct employee support.

**Therefore, the most relevant options are A and C.**

**Final Answer: A and C**

### TODO - REFLECT 🖊

🖊 TODO: Explain how the redesigned output differs from the two decision-oriented outputs.

🖊 TODO: Identify one useful evidence item and one item that still requires verification.

🖊 TODO: Explain how the redesign preserves human judgment and accountability.


# Part 3: Business Outcome and Simpler Alternative

Choose one Part 1 project. Start with the result the business needs, not the technology.


### TODO - DECIDE 🖊

Choose **one** project from Part 1.

**Selected project:** 🖊 TODO

**Business outcome:** 🖊 TODO: State the result that should improve.

**Metric, baseline, and target:** 🖊 TODO: Name one metric, the baseline evidence needed, and a proposed target.

**Simpler alternative:** 🖊 TODO: Identify a rule, conventional analysis, training intervention, or process change.

**Comparison evidence:** 🖊 TODO: Explain how the business could compare the simpler alternative with the AI proposal.

**Initial recommendation:** 🖊 TODO: Test the simpler approach first, pilot AI, combine them, or do not proceed—and explain why.


# Part 4: Reflect on the Assignment

Integrate what you observed across classification, résumé review, evidence gathering, and process fit.


### TODO - FINAL REFLECTION 🖊

Write **150–200 words** addressing all four questions:

1. What did the model do well and poorly in the classifications?
2. What did the résumé prompts reveal about unsupported precision or authority?
3. How would you use AI to gather evidence without delegating the employment decision?
4. Why compare an AI proposal with a simpler process change and measurable outcome?

🖊 TODO: Write your reflection here.
